In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [22]:
import numpy as np
import pandas as pd
import os
import warnings
import joblib
import gc

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

import xgboost as xgb
import mlflow
import dagshub

warnings.filterwarnings("ignore")
print("All imports OK ✓")

All imports OK ✓


**load and merge data. clean, reduce memory**

In [23]:
print("Setting up MLflow + DagsHub...")
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "xgboost_experiments"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
    print(f"Created experiment: {experiment_name}")
else:
    print(f"Found existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)
print("MLflow + DagsHub connected ✓")

Setting up MLflow + DagsHub...


Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

Found existing experiment: xgboost_experiments
MLflow + DagsHub connected ✓


In [32]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading data...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")
print(f"  train_transaction: {train_tr.shape}")
print(f"  train_identity:    {train_id.shape}")
print(f"  test_transaction:  {test_tr.shape}")
print(f"  test_identity:     {test_id.shape}")

print("\nFixing column names (hyphen → underscore)...")
train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

print("\nMerging...")
train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"  train merged: {train.shape}")
print(f"  test merged:  {test.shape}")

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

print("\nAligning columns...")
only_in_train = set(X.columns) - set(test.columns)
only_in_test  = set(test.columns) - set(X.columns)
print(f"  Cols only in train: {len(only_in_train)}")
print(f"  Cols only in test:  {len(only_in_test)}")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
print(f"  Aligned: X={X.shape}, test={test.shape}")
print(f"  Fraud rate: {y.mean():.4f} ({y.sum()} fraud out of {len(y)} total)")

Loading data...
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  test_transaction:  (506691, 393)
  test_identity:     (141907, 41)

Fixing column names (hyphen → underscore)...

Merging...
  train merged: (590540, 434)
  test merged:  (506691, 433)

Aligning columns...
  Cols only in train: 0
  Cols only in test:  0
  Aligned: X=(590540, 432), test=(506691, 432)
  Fraud rate: 0.0350 (20663 fraud out of 590540 total)


**feature engineering**

In [33]:
print("=== FEATURE ENGINEERING ===")
print(f"Starting shape: X={X.shape}, test={test.shape}")
cols_before_engineering = X.shape[1]

def engineer_features(df):
    df["TransactionAmt_log"]      = np.log1p(df["TransactionAmt"]).astype(np.float32)
    df["TransactionAmt_is_round"] = (df["TransactionAmt"] % 1 == 0).astype(np.int8)
    df["TransactionAmt_cents"]    = (df["TransactionAmt"] % 1).astype(np.float32)
    df["hour"] = ((df["TransactionDT"] / 3600) % 24).astype(np.float32)
    df["day"]  = ((df["TransactionDT"] / 86400) % 7).astype(np.float32)
    if "P_emaildomain" in df.columns:
        df["P_email_provider"] = df["P_emaildomain"].str.split(".").str[0].fillna("unknown")
    if "R_emaildomain" in df.columns:
        df["R_email_provider"] = df["R_emaildomain"].str.split(".").str[0].fillna("unknown")
    df["card_identity"] = (
        df["card1"].astype(str) + "_" +
        df["card2"].astype(str) + "_" +
        df["card4"].astype(str)
    )
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    if len(c_cols) > 0:
        print(f"  C columns found: {c_cols}")
        df["C_mean"] = df[c_cols].mean(axis=1).astype(np.float32)
        df["C_std"]  = df[c_cols].std(axis=1).astype(np.float32)
        df["C_max"]  = df[c_cols].max(axis=1).astype(np.float32)
        df["C_min"]  = df[c_cols].min(axis=1).astype(np.float32)

print("Computing card1 aggregates from train...")
card1_count    = X["card1"].value_counts()
card1_amt_mean = X.groupby("card1")["TransactionAmt"].median()
card1_amt_std  = X.groupby("card1")["TransactionAmt"].std()
print(f"  Unique card1 values: {len(card1_count)}")

def add_card_aggregates(df):
    df["card1_count"]    = df["card1"].map(card1_count).astype(np.float32)
    df["card1_amt_mean"] = df["card1"].map(card1_amt_mean).astype(np.float32)
    df["card1_amt_std"]  = df["card1"].map(card1_amt_std).astype(np.float32)

print("Engineering X features...")
engineer_features(X)
add_card_aggregates(X)
gc.collect()
print(f"  X shape: {X.shape}, memory: {X.memory_usage(deep=True).sum()/1024**2:.1f} MB")

print("Engineering test features...")
engineer_features(test)
add_card_aggregates(test)
gc.collect()
print(f"  test shape: {test.shape}, memory: {test.memory_usage(deep=True).sum()/1024**2:.1f} MB")

print("\nRealigning columns...")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)

cols_after_engineering = X.shape[1]
print(f"Final: X={X.shape}, test={test.shape}")
print(f"New cols added: {cols_after_engineering - cols_before_engineering}")
print("Feature engineering done ✓")

=== FEATURE ENGINEERING ===
Starting shape: X=(590540, 432), test=(506691, 432)
Computing card1 aggregates from train...
  Unique card1 values: 13553
Engineering X features...
  C columns found: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  X shape: (590540, 447), memory: 2630.0 MB
Engineering test features...
  C columns found: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  test shape: (506691, 447), memory: 2267.5 MB

Realigning columns...
Final: X=(590540, 447), test=(506691, 447)
New cols added: 15
Feature engineering done ✓


**feature selection**

In [34]:
print("=== FEATURE SELECTION ===")
print(f"Starting shape: X={X.shape}, test={test.shape}")

# ── 1. drop high missing >90% ──
print("\nStep 1: Dropping columns with >90% missing...")
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()
print(f"  Dropped: {len(high_missing)} cols")
print(f"  {high_missing}")
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)
print(f"  After: X={X.shape}")

# ── 2. drop near-zero variance ──
print("\nStep 2: Dropping near-zero variance (std < 0.01)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"  Dropped: {len(low_var)} cols → {low_var}")
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)
print(f"  After: X={X.shape}")

# ── 3. drop raw C columns ──
print("\nStep 3: Dropping raw C columns...")
c_cols_dropped = [c for c in X.columns if c.startswith("C") and c[1:].isdigit()]
print(f"  Dropping: {c_cols_dropped}")
X    = X.drop(columns=c_cols_dropped)
test = test.drop(columns=c_cols_dropped)
print(f"  After: X={X.shape}")

# ── 4. drop highly correlated (r > 0.99) ──
print("\nStep 4: Dropping highly correlated features (r > 0.99)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = X[num_cols_temp].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if any(upper[col] > 0.99)]
print(f"  Dropped: {len(high_corr)} cols")
print(f"  {high_corr}")
X    = X.drop(columns=high_corr)
test = test.drop(columns=high_corr)
print(f"  After: X={X.shape}")

# ── 5. drop high cardinality categoricals ──
print("\nStep 5: Dropping high cardinality categoricals (>200 unique)...")
cat_cols_temp = X.select_dtypes(include=["object"]).columns
high_card = [c for c in cat_cols_temp if X[c].nunique() > 200]
print(f"  Dropped: {len(high_card)} cols → {high_card}")
X    = X.drop(columns=high_card)
test = test.drop(columns=high_card)
print(f"  After: X={X.shape}")

print(f"\n=== FINAL SHAPES AFTER SELECTION ===")
print(f"  X:    {X.shape}")
print(f"  test: {test.shape}")
assert list(X.columns) == list(test.columns)
print("Columns aligned ✓")

# save these for logging later
cols_after_cleaning = X.shape[1]
final_feature_count = X.shape[1]
final_cols          = X.columns.tolist()
print(f"\ncols_after_engineering: {cols_after_engineering}")
print(f"cols_after_cleaning:    {cols_after_cleaning}")
print(f"final_feature_count:    {final_feature_count}")

=== FEATURE SELECTION ===
Starting shape: X=(590540, 447), test=(506691, 447)

Step 1: Dropping columns with >90% missing...
  Dropped: 12 cols
  ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']
  After: X=(590540, 435)

Step 2: Dropping near-zero variance (std < 0.01)...
  Dropped: 3 cols → ['V1', 'V305', 'C_min']
  After: X=(590540, 432)

Step 3: Dropping raw C columns...
  Dropping: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  After: X=(590540, 418)

Step 4: Dropping highly correlated features (r > 0.99)...
  Dropped: 39 cols
  ['D12', 'V18', 'V97', 'V101', 'V102', 'V103', 'V132', 'V133', 'V134', 'V164', 'V167', 'V177', 'V179', 'V182', 'V211', 'V213', 'V231', 'V232', 'V233', 'V269', 'V276', 'V279', 'V293', 'V295', 'V298', 'V306', 'V316', 'V317', 'V318', 'V322', 'V323', 'V324', 'V329', 'V330', 'V331', 'V332', 'V333', 'hour', 'C_max']
  After: X=(590540, 379)

Step 5: Dropping high 

In [35]:
print("=== MEMORY REDUCTION ===")
print(f"Before: X={X.memory_usage(deep=True).sum()/1024**2:.1f} MB, test={test.memory_usage(deep=True).sum()/1024**2:.1f} MB")

def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")

print(f"\nAfter:")
print(f"  X:    {X.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"  test: {test.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nColumn dtypes summary:")
print(X.dtypes.value_counts())

=== MEMORY REDUCTION ===
Before: X=2224.6 MB, test=1919.1 MB
  X: 1462.0 MB
  test: 1264.8 MB

After:
  X:    1462.0 MB
  test: 1264.8 MB

Column dtypes summary:
float32    344
object      29
int32        1
int16        1
int8         1
Name: count, dtype: int64


**MLflow / DagsHub setup**

**Split**

In [36]:
print("=== TRAIN/VAL SPLIT ===")
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"  Train fraud rate: {y_train.mean():.4f}")
print(f"  Val   fraud rate: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"\n  Numeric cols:     {len(num_cols)}")
print(f"  Categorical cols: {len(cat_cols)}")
print(f"  Cat cols are:     {cat_cols}")

del X
gc.collect()
print("\nFreed X from memory ✓")

=== TRAIN/VAL SPLIT ===
  X_train: (472432, 376), y_train: (472432,)
  X_val:   (118108, 376),   y_val:   (118108,)
  Train fraud rate: 0.0350
  Val   fraud rate: 0.0350

  Numeric cols:     347
  Categorical cols: 29
  Cat cols are:     ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'P_email_provider', 'R_email_provider']

Freed X from memory ✓


**Build XGBoost pipeline & train**

In [37]:
print("=== FEATURE SELECTION VIA XGBOOST IMPORTANCE ===")

# use best params from search
best_params = {
    "subsample": 0.9,
    "reg_lambda": 0.5,
    "reg_alpha": 0.5,
    "n_estimators": 500,
    "min_child_weight": 50,
    "max_depth": 6,
    "learning_rate": 0.5,
    "colsample_bytree": 0.6
}

# step 1 — fit preprocessor only
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

print("Fitting preprocessor...")
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
feature_names = num_cols + cat_cols
print(f"  Processed shape: {X_train_processed.shape}")

# step 2 — train XGBoost on all features
print("\nTraining XGBoost on all features...")
fraud_count      = int(y_train.sum())
nonfraud_count   = int((y_train == 0).sum())
scale_pos_weight = nonfraud_count / fraud_count

xgb_model = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="auc",
    tree_method="hist",
    device="cuda" if os.path.exists("/usr/local/cuda") else "cpu",
    n_jobs=-1
)
xgb_model.fit(X_train_processed, y_train)

val_probs = xgb_model.predict_proba(X_val_processed)[:, 1]
val_auc_all = roc_auc_score(y_val, val_probs)
print(f"  Val AUC (all features): {val_auc_all:.4f}")

# step 3 — get feature importances
importances = pd.Series(xgb_model.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=False)

print(f"\nTop 20 most important features:")
print(importances.head(20))
print(f"\nBottom 20 least important features:")
print(importances.tail(20))

# step 4 — try different thresholds
print("\n=== TESTING DIFFERENT FEATURE THRESHOLDS ===")
thresholds = [0.0001, 0.0005, 0.001, 0.005]

best_threshold = None
best_threshold_auc = 0
threshold_results = []

for thresh in thresholds:
    selected = importances[importances >= thresh].index.tolist()
    selected_idx = [feature_names.index(f) for f in selected]

    X_train_sel = X_train_processed[:, selected_idx]
    X_val_sel   = X_val_processed[:, selected_idx]

    m = xgb.XGBClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1
    )
    m.fit(X_train_sel, y_train)
    probs = m.predict_proba(X_val_sel)[:, 1]
    auc   = roc_auc_score(y_val, probs)

    threshold_results.append({
        "threshold": thresh,
        "n_features": len(selected),
        "val_auc": auc
    })
    print(f"  threshold={thresh}: {len(selected)} features → Val AUC={auc:.4f}")

    if auc > best_threshold_auc:
        best_threshold_auc = auc
        best_threshold     = thresh

print(f"\nBest threshold: {best_threshold} → AUC {best_threshold_auc:.4f}")

# step 5 — retrain final pipeline with selected features only
print("\nRetraining final pipeline with selected features...")
selected_features = importances[importances >= best_threshold].index.tolist()
print(f"  Selected {len(selected_features)} features out of {len(feature_names)}")

selected_num_cols = [c for c in selected_features if c in num_cols]
selected_cat_cols = [c for c in selected_features if c in cat_cols]
print(f"  Numeric: {len(selected_num_cols)}, Categorical: {len(selected_cat_cols)}")

final_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),          selected_num_cols),
    ("cat", Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), selected_cat_cols)
])

model_xgb_final = Pipeline(steps=[
    ("preprocessor", final_preprocessor),
    ("classifier",   xgb.XGBClassifier(
        **best_params,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",
        device="cuda" if os.path.exists("/usr/local/cuda") else "cpu",
        n_jobs=-1
    ))
])

model_xgb_final.fit(X_train, y_train)

val_probs_final = model_xgb_final.predict_proba(X_val)[:, 1]
val_auc_final   = roc_auc_score(y_val, val_probs_final)
print(f"\n  Val AUC (selected features): {val_auc_final:.4f}")
print(f"  Val AUC (all features):      {val_auc_all:.4f}")
print(f"  Difference:                  {val_auc_final - val_auc_all:+.4f}")

# step 6 — log to MLflow
print("\nLogging to MLflow...")
with mlflow.start_run(run_name="XGBoost_Feature_Importance_Selection"):
    mlflow.log_param("best_threshold",    best_threshold)
    mlflow.log_param("features_before",   len(feature_names))
    mlflow.log_param("features_after",    len(selected_features))
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        "val_auc_all_features":      val_auc_all,
        "val_auc_selected_features": val_auc_final,
    })
    # save feature importance as artifact
    importances.to_csv("feature_importances.csv")
    mlflow.log_artifact("feature_importances.csv")

    mlflow.sklearn.log_model(
        model_xgb_final,
        artifact_path="xgboost_pipeline_selected",
        registered_model_name="XGBoost_Fraud_Pipeline_Selected"
    )
    print(f"  Run ID: {mlflow.active_run().info.run_id}")

print("\nDone ✓")

=== FEATURE SELECTION VIA XGBOOST IMPORTANCE ===
Fitting preprocessor...
  Processed shape: (472432, 376)

Training XGBoost on all features...
  Val AUC (all features): 0.9627

Top 20 most important features:
V218         0.124943
V257         0.105821
V201         0.102396
id_35        0.058879
V294         0.036585
V244         0.030463
V91          0.029786
V70          0.025267
V149         0.013457
V258         0.012444
ProductCD    0.010417
V48          0.007777
V79          0.006814
card6        0.006107
V154         0.005884
V285         0.005694
V303         0.005516
V155         0.005504
V283         0.005313
V45          0.004901
dtype: float32

Bottom 20 least important features:
V252    0.0
V68     0.0
V65     0.0
V256    0.0
V284    0.0
V238    0.0
V240    0.0
V249    0.0
V235    0.0
V228    0.0
V237    0.0
V241    0.0
V247    0.0
V327    0.0
V41     0.0
V325    0.0
V27     0.0
V28     0.0
V14     0.0
M1      0.0
dtype: float32

=== TESTING DIFFERENT FEATURE THRESHOLDS ==

2026/05/03 12:51:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:51:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_Fraud_Pipeline_Selected'.
2026/05/03 12:51:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Fraud_Pipeline_Selected, version 1
Created version '1' of model 'XGBoost_Fraud_Pipeline_Selected'.


  Run ID: 293c159ab20344f5a8a2627f20a8698e
🏃 View run XGBoost_Feature_Importance_Selection at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/293c159ab20344f5a8a2627f20a8698e
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/2

Done ✓


In [38]:
print("=== HYPERPARAMETER SEARCH SUMMARY ===")
results_df = pd.DataFrame(search.cv_results_)
results_df["params_clean"] = results_df["params"].apply(
    lambda p: {k.replace("classifier__", ""): v for k, v in p.items()}
)
results_df = results_df.sort_values("mean_test_score", ascending=False)

print(f"\nTop 10 trials (sorted by CV AUC):")
print(f"{'Rank':<6} {'CV AUC':>8} {'Std':>8}  Params")
print("-" * 80)
for _, row in results_df.iterrows():
    print(f"  {int(row['rank_test_score']):<4} {row['mean_test_score']:.4f}   ±{row['std_test_score']:.4f}  {row['params_clean']}")

print(f"\n=== BEST MODEL ===")
print(f"  CV AUC:  {search.best_score_:.4f}")
print(f"  Val AUC: {val_auc:.4f}")
print(f"  Params:")
for k, v in search.best_params_.items():
    print(f"    {k.replace('classifier__', '')}: {v}")

=== HYPERPARAMETER SEARCH SUMMARY ===

Top 10 trials (sorted by CV AUC):
Rank     CV AUC      Std  Params
--------------------------------------------------------------------------------
  1    0.9409   ±0.0005  {'subsample': 0.9, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 50, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.6}
  2    0.9267   ±0.0013  {'subsample': 0.6, 'reg_lambda': 5.0, 'reg_alpha': 0.5, 'n_estimators': 300, 'min_child_weight': 100, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  3    0.9145   ±0.0022  {'subsample': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
  4    0.9040   ±0.0026  {'subsample': 0.6, 'reg_lambda': 1.0, 'reg_alpha': 0.01, 'n_estimators': 100, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
  5    0.8983   ±0.0033  {'subsample': 0.8, 'reg_lamb

**Evaluate & log**

**Submission**

In [39]:
print("=== GENERATING SUBMISSION FOR BEST MODEL ===")
print(f"Model: XGBoost with 184 selected features, Val AUC=0.9656")

test_probs = model_xgb_final.predict_proba(test)[:, 1]

print(f"  Predictions shape: {test_probs.shape}")
print(f"  Predicted fraud rate: {test_probs.mean():.4f}")
print(f"  Min: {test_probs.min():.4f}, Max: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

submission.to_csv("submission_xgb_feature_selected.csv", index=False)
print("\nsubmission_xgb_feature_selected.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION FOR BEST MODEL ===
Model: XGBoost with 184 selected features, Val AUC=0.9656
  Predictions shape: (506691,)
  Predicted fraud rate: 0.0680
  Min: 0.0000, Max: 1.0000

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.017008
1        3663550  0.005214
2        3663551  0.003359
3        3663552  0.000099
4        3663553  0.000028

submission_xgb_feature_selected.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
